In [1]:
import os

from agent_framework import Agent, MCPStreamableHTTPTool
from agent_framework.openai import OpenAIChatOptions
from dotenv import load_dotenv

from azure_client import create_chat_client

load_dotenv()

True

In [2]:
tool = MCPStreamableHTTPTool(
    name="Microsoft Learn MCP",
    url=os.getenv("MCP_LEARN_URL", "https://learn.microsoft.com/api/mcp"),
    # we don't require approval for microsoft_docs_search tool calls
    # but we do for any other tool
    # approval_mode={"never_require_approval": ["microsoft_docs_search"]},
)

In [3]:
# Entra ID auth against Azure Gov; see azure_client.py
llm = create_chat_client()

agent = Agent(
    llm,
    "You are a helpful agent. You use Model Context Protocol (MCP) tools to answer user questions. "
    "You can only respond using the tools available to you. Do not make up tool functionality. The tools will be"
    "Provided to you in the prompt.",
    name="test_agent",
    tools=[tool],
    default_options=OpenAIChatOptions(temperature=0),
)

query = "Search the docs and tell me what MCP is."
print(f"User: {query}")
print("\n=== HTTP Request/Response Details Below ===\n")
result = await agent.run(query)
print("\n=== End of HTTP Details ===\n")
print(f"Agent: {result}\n")

User: Search the docs and tell me what MCP is.

=== HTTP Request/Response Details Below ===




=== End of HTTP Details ===

Agent: MCP stands for **Model Context Protocol**.

From the Microsoft documentation:

- MCP is **an open protocol / open standard** that defines a **standard way for AI agents and LLM-based applications to connect to external tools, data sources, and services**.
- It provides a **client–server protocol**:  
  - An **MCP server** exposes tools, resources, prompts, and data.  
  - An **MCP client** (for example, an AI assistant or agent) discovers those tools and calls them as needed.
- The goal is to **standardize integrations** so you don’t need custom, one-off APIs for every app. Instead, any MCP‑compatible agent can talk to any MCP server in a consistent way.

Microsoft uses MCP in many places, for example:

- **Windows**: “MCP on Windows” uses the On-device Agent Registry so agents can securely discover and use MCP servers from local apps and remote services, with containment, permissions, and auditing.  
- **Databricks**: Databricks exposes **Managed, 

In [4]:
from random import randint
from typing import Annotated


def get_weather(
    location: Annotated[str, "The location to get the weather for."],
) -> str:
    """Get the weather for a given location."""
    conditions = ["sunny", "cloudy", "rainy", "stormy"]
    return f"The weather in {location} is {conditions[randint(0, 3)]} with a high of {randint(10, 30)}°C."


weather_agent = Agent(
    llm,
    "You are a helpful weather agent. Use the tools available to you to answer questions.",
    name="weather_agent",
    tools=[get_weather],
    default_options=OpenAIChatOptions(temperature=0),
)


async def non_streaming_example(agent: Agent, query: str) -> None:
    """Example of non-streaming response (get the complete result at once)."""
    print("=== Non-streaming Response Example ===")

    print(f"User: {query}")
    result = await agent.run(query)
    print(f"Agent: {result}\n")


async def streaming_example(agent: Agent, query: str) -> None:
    """Example of streaming response (get results as they are generated)."""
    print("=== Streaming Response Example ===")

    print(f"User: {query}")
    print("Agent: ", end="", flush=True)
    async for chunk in agent.run(query, stream=True):
        if chunk.text:
            print(chunk.text, end="", flush=True)
    print("\n")

In [5]:
await streaming_example(weather_agent, "What's the weather like in Portland?")

=== Streaming Response Example ===
User: What's the weather like in Portland?
Agent: 

The

 weather

 in

 Portland

...

 high

 of

23

°C

.

In [6]:
# MCP transports must be closed in the task that opened them. Skipping this leaves the
# connection to be finalized by the garbage collector, which raises
# "Attempted to exit cancel scope in a different task than it was entered in".
if tool.is_connected:
    await tool.close()
    print(f"closed {tool.name}")


closed Microsoft Learn MCP
